# Spearman — **top3 vs top5** comparison

**Standalone** notebook: it reads the CSVs already produced (no recomputation) and compares the two
versions of the Spearman analysis. Significance: **p rounded to 2 decimals <= 0.05** (0.054 included).

Spearman is computed over *all* patches of a slide (it has no K): the dimensions are **tumor**,
**signature**, **score** (grad / combined) and **contrast** (`TP<TN`, `pred<TN`).


## 0. Config e caricamento delle due versioni

In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

OUT_DIR = '../grad/xai_outputs'
# {label: file name}.  The "non top5" version is the one without a suffix.
VERSIONI = {'top5': 'spearman_specificity_ALL_top5.csv',
            'top3': 'spearman_specificity_ALL.csv'}
NMIN = 10          # below this group size the p-value is not reliable
PSIG = lambda s: s.round(2) <= 0.05

parts = []
for ver, fn in VERSIONI.items():
    f = os.path.join(OUT_DIR, fn)
    if not os.path.exists(f):
        print(f'[{ver}] MANCANTE: {f}'); continue
    t = pd.read_csv(f); t['versione'] = ver
    t['MWU_p'] = pd.to_numeric(t['MWU_p'], errors='coerce')
    parts.append(t)
    print(f'[{ver}] {fn}: {len(t)} righe')
D = pd.concat(parts, ignore_index=True)
D['n_min']  = D[['nA', 'nB']].min(axis=1)
D['effect'] = -D['delta']                      # Spearman: delta negativo = piu' specifico
D['negp']   = -np.log10(D['MWU_p'].clip(lower=1e-6))
D['sig']    = PSIG(D['MWU_p'])
D['lab']    = D['tissue'].str.upper() + ' ' + D['target']
D['key']    = D['lab'] + ' | ' + D['score'].str.replace('_score', '', regex=False) + ' | ' + D['contrast']

print('\nsignificative per versione:')
display(D[D['sig']].groupby(['versione', 'score', 'contrast']).size().rename('n').reset_index())

## 1. What is significant in which version
Every combination (tumor, signature, score, contrast) is classified as significant in **both**,
in **top3** only, in **top5** only, or present in a single version (not comparable).


In [ ]:
piv  = D.pivot_table(index='key', columns='versione', values='MWU_p', aggfunc='min')
sigp = D.pivot_table(index='key', columns='versione', values='sig', aggfunc='max').fillna(False)
vers = [v for v in ['top3', 'top5'] if v in piv.columns]

def _cls(r):
    pres = [v for v in vers if pd.notna(piv.loc[r.name, v])]
    if len(pres) < 2:                       return f'solo in {pres[0]} (non confrontabile)' if pres else '-'
    s3, s5 = bool(r.get('top3', False)), bool(r.get('top5', False))
    return ('entrambe' if s3 and s5 else 'solo top3' if s3 else 'solo top5' if s5 else 'nessuna')

cmp_ = pd.DataFrame({'classe': sigp.apply(_cls, axis=1)})
for v in vers: cmp_[f'p_{v}'] = piv[v].round(4)
cmp_ = cmp_[cmp_['classe'] != 'nessuna'].sort_values(['classe', f'p_{vers[-1]}'])

print('riepilogo:')
display(cmp_['classe'].value_counts().rename('n').reset_index())
pd.set_option('display.max_rows', 300)
display(cmp_)

## 2. Scatter dei p-value: top3 vs top5
Ogni punto e' una combinazione presente in **entrambe**. Assi in -log10(p): piu' in alto/a destra =
piu' significativo. Le linee tratteggiate sono la soglia; la diagonale indica dove le due versioni
darebbero lo stesso risultato.

In [ ]:
both = piv.dropna(subset=vers) if len(vers) == 2 else pd.DataFrame()
if both.empty:
    print('serve che entrambe le versioni contengano le stesse combinazioni')
else:
    x = -np.log10(both['top3'].clip(lower=1e-6)); y = -np.log10(both['top5'].clip(lower=1e-6))
    col = np.where(PSIG(both['top3']) & PSIG(both['top5']), '#2ca02c',
          np.where(PSIG(both['top3']), '#1f77b4',
          np.where(PSIG(both['top5']), '#ff7f0e', '#cccccc')))
    fig, ax = plt.subplots(figsize=(6.4, 6))
    lim = max(x.max(), y.max()) * 1.08
    ax.plot([0, lim], [0, lim], color='0.7', lw=0.8, ls='-', zorder=1)
    thr = -np.log10(0.055)
    ax.axhline(thr, color='0.6', ls='--', lw=0.8); ax.axvline(thr, color='0.6', ls='--', lw=0.8)
    ax.scatter(x, y, c=col, s=38, edgecolor='0.25', linewidth=0.4, zorder=3)
    for k in both.index[(PSIG(both['top3']) | PSIG(both['top5']))][:18]:
        ax.annotate(k.split(' | ')[0], (x[k], y[k]), fontsize=6,
                    xytext=(3, 3), textcoords='offset points')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel('-log10(p)  —  top3'); ax.set_ylabel('-log10(p)  —  top5')
    ax.set_title('Concordanza dei p-value fra le due versioni', fontsize=12)
    for sp_ in ('top', 'right'): ax.spines[sp_].set_visible(False)
    ax.legend(handles=[Line2D([], [], marker='o', ls='', color=c, markersize=7, label=l)
                       for c, l in [('#2ca02c', 'sig. in entrambe'), ('#1f77b4', 'solo top3'),
                                    ('#ff7f0e', 'solo top5'), ('#cccccc', 'in nessuna')]],
              frameon=False, fontsize=8, loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'spearman_confronto_scatter.png'), dpi=300,
                bbox_inches='tight', facecolor='white')
    plt.show()

## 3. Tumor x signature grid, side by side per version
One panel per version (same rows and columns, so they can be compared at a glance).
**Shape** = score, **colour** = -log10(p), **red outline** = one group has fewer than `NMIN` cases.


In [ ]:
S = D[D['sig']].copy()
CONTR = 'pred<TN'                     # cambia in 'TP<TN' per l'altro contrasto
S = S[S['contrast'] == CONTR]
if S.empty:
    print(f'nessuna combinazione significativa per {CONTR}')
else:
    SCORES = sorted(S['score'].unique()); mk = {s: m for s, m in zip(SCORES, ['o', 's', '^', 'D'])}
    tums = sorted(S['tissue'].str.upper().unique())
    sigs = S.groupby('target').size().sort_values().index.tolist()
    xi = {t: i for i, t in enumerate(tums)}; yi = {s: i for i, s in enumerate(sigs)}
    vmax = float(S['negp'].max()); cmap = plt.cm.viridis
    plt.rcParams.update({'font.size': 9})

    fig, axes = plt.subplots(1, len(vers), figsize=(len(vers) * (0.42 * len(tums) + 3.4),
                                                    0.32 * len(sigs) + 2.6), squeeze=False)
    for ax, ver in zip(axes[0], vers):
        sub = S[S['versione'] == ver]
        for sc in SCORES:
            s2 = sub[sub['score'] == sc]
            if s2.empty: continue
            off = (SCORES.index(sc) - (len(SCORES) - 1) / 2) * 0.30
            ax.scatter(s2['tissue'].str.upper().map(xi) + off, s2['target'].map(yi),
                       c=s2['negp'], cmap=cmap, vmin=0, vmax=vmax, marker=mk[sc], s=62,
                       edgecolor=np.where(s2['n_min'] < NMIN, 'crimson', '0.25'),
                       linewidth=np.where(s2['n_min'] < NMIN, 1.2, 0.5), zorder=3)
        ax.set_xticks(range(len(tums))); ax.set_xticklabels(tums, rotation=90, fontsize=7)
        ax.set_yticks(range(len(sigs))); ax.set_yticklabels(sigs, fontsize=7)
        ax.set_xlim(-0.7, len(tums) - 0.3); ax.set_ylim(-0.8, len(sigs) - 0.2)
        ax.grid(color='0.92', lw=0.5, zorder=0)
        for sp_ in ('top', 'right'): ax.spines[sp_].set_visible(False)
        ax.set_title(f'{ver}   (n = {len(sub)})', fontsize=11)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, vmax))
    fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.015, pad=0.015).set_label('-log10(p-value)')
    handles = [Line2D([], [], marker=mk[s], ls='', color='0.35', markersize=7,
                      label=s.replace('_score', '')) for s in SCORES]
    handles.append(Line2D([], [], marker='o', ls='', markerfacecolor='0.75',
                          markeredgecolor='crimson', markeredgewidth=1.2, markersize=7,
                          label=f'gruppo < {NMIN}'))
    axes[0][-1].legend(handles=handles, title='score', loc='upper left',
                       bbox_to_anchor=(1.02, 1), frameon=False, fontsize=8)
    fig.suptitle(f'Spearman — contrasto {CONTR}: top3 vs top5', fontsize=13, y=1.01)
    plt.savefig(os.path.join(OUT_DIR, f'spearman_griglia_confronto_{CONTR.replace("<","lt")}.png'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

## 4. Tabella finale e salvataggio

In [ ]:
cols = ['versione', 'tissue', 'target', 'score', 'contrast', 'nA', 'nB',
        'delta', 'effect', 'MWU_p', 'n_min']
tab = D[D['sig']].sort_values(['MWU_p'])[cols].reset_index(drop=True)
display(tab)
tab.to_csv(os.path.join(OUT_DIR, 'spearman_significativi_confronto.csv'), index=False)
cmp_.to_csv(os.path.join(OUT_DIR, 'spearman_confronto_top3_top5.csv'))
print('salvati: spearman_significativi_confronto.csv  e  spearman_confronto_top3_top5.csv')

## 5. How to read it
- **Cell 1**: the classification tells you whether a result holds in *both* versions. The
  **"both"** combinations are the most credible: they do not depend on the threshold used to define
  the prediction.
- **Cell 2 (scatter)**: points above the diagonal are more significant in top5, below it in top3. A
  cloud around the diagonal means the top3/top5 choice matters little.
- **Caution**: top5 marks more signatures as "predicted", so the **TN group shrinks**. Points with a
  red outline in top5 but not in top3 are probably an artefact of that shrinkage rather than extra
  signal.
- Combinations present in only one version are not comparable: the two files cover different sets.
